# SECOM data audit

This notebook checks the public source files before any imputation or process interpretation. It keeps the anonymous feature names and treats each row as a production example, not automatically as a wafer.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.secom_data import load_secom, audit_secom

## Load and validate

The loader checks that feature, label, and timestamp rows align. If it raises an error, inspect the source files rather than silently dropping rows.

In [ ]:
dataset = load_secom(ROOT / 'data' / 'raw')
audit = audit_secom(dataset)
{key: value for key, value in audit.items() if key != 'feature_profile'}

## Inspect missingness and feature variation

A constant or near-constant flag is a prompt for review, not an automatic deletion rule. A high missing-value count is also a reason to investigate before choosing imputation.

In [ ]:
feature_profile = audit['feature_profile']
feature_profile.sort_values(['missing_fraction', 'dominant_value_fraction'], ascending=False).head(20)

## Interpretation notes

- Confirm the label mapping and timestamp order from the source documentation.
- Do not name anonymized measurements as temperature, pressure, or another physical variable.
- Do not treat timestamps alone as proof that every row is a continuous sample from one stable process.
- Record preprocessing decisions before using charts or models.